## Running VGAE / Graph-PCA & baseline models

- Dim-reduction on features ($x \in \mathbb{R}^{N \times P}$: observation $z \in \mathbb{R}^{N \times D}$: representation, $u \in \mathbb{R}^{N \times 1}$: interpretability as "trajectory / gradient")
- Reconstruction w/ $\sigma(z \cdot z^T)$ for graph reconstruction, regularize $u$ w/ CyCIF graph prior
- Benchmark against K-means clustering & PCA, regular VAE, etc.

In [ ]:
import os
import gc
import sys
import pickle

import numpy as np
import cupy as cp
import pandas as pd
import scanpy as sc

import torch
import tifffile
import torch.nn as nn

import networkx as nx
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy import sparse
from torch_geometric import utils as pyg_utils

torch.manual_seed(42)

In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, plot, gen_graph, configs, dataset, zonation
import baseline, vgae_sbm, gpca, model_train, model_eval

from torch_geometric.loader import DataLoader

### baseline VAE / ConvVAE

In [ ]:
# Load DESI normed image
data_path = '../data/integrated/'
desi_imgs = [tifffile.imread(os.path.join(data_path, 'desi', f))
             for f in sorted(os.listdir(os.path.join(data_path, 'desi')))
             if f[-3:] == 'tif']

# Load full-size u_prior from CyCIF image (graph-based heat diffusion)
u_priors = [tifffile.imread(os.path.join(data_path, 'zonation_prior', f))
            for f in sorted(os.listdir(os.path.join(data_path, 'zonation_prior')))
            if f[-3:] == 'tif']

nchans, ndimy, ndimx = desi_imgs[0].shape

In [ ]:
ims_data = dataset.DESIDataset(data_path='../data/integrated/desi_debug/DESI_slice_48.ome.tif',
                               prior_path='../data/integrated/zonation_prior_debug/DESI_slice_48_prior.tif')

ims_train_dl = DataLoader(ims_data, batch_size=512, shuffle=True)

In [ ]:
# DEBUG model
from importlib import reload
reload(baseline)
reload(vgae_sbm)
reload(utils)
reload(plot)
reload(model_train)
reload(model_eval)
reload(configs)
reload(dataset)

In [ ]:
lr = 1e-2
n_epochs = 100
device = torch.device('cuda')

train_configs = configs.set_train_configs(data_path=None, 
                                          n_epochs=n_epochs, 
                                          gamma=0.99,
                                          lr=lr)

model_configs = configs.set_model_configs(device=device,
                                          beta=0.5,
                                          c_in=nchans,
                                          c_hidden=5,
                                          c_latent=1,
                                          pu_scale=0.5)

In [ ]:
model = baseline.VAE(model_configs)
losses, nlls, kls, ols = model.model_train(train_configs=train_configs, dataloader=ims_train_dl)

In [ ]:
idx = 10
pu = u_priors[idx]
desi_feature_mat = desi_imgs[idx].transpose(2, 1, 0).reshape(-1, nchans)
latent, recon = model.model_eval(desi_feature_mat)
px = recon.px_mu.detach().cpu().numpy()  # [Y*X, C]
px_chans = px.reshape(ndimy, ndimx, nchans).transpose(2, 0, 1) # [C, Y, X]
qu = latent.qu.detach().cpu().squeeze().numpy().reshape(ndimy, ndimx).T  # [Y, X]
qu_discrete = utils.infer_zones(qu, nbins=10)

In [ ]:
sample_idx = np.random.choice(np.arange(desi_feature_mat.shape[0]), 500, replace=False)

fig, ax = plt.subplots()
ax.scatter(desi_feature_mat[sample_idx, :].flatten(), px[sample_idx, :].flatten(), s=0.1)
ax.spines[['right', 'top']].set_visible(False)
ax.get_xaxis().tick_bottom()
ax.get_yaxis().tick_left()
ax.set_title('Feature matrix reconstruction')
plt.show()

del sample_idx
gc.collect()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
ax1.imshow(pu, cmap='jet')
ax1.axis('off')
ax1.set_title('U prior')
ax2.imshow(qu, cmap='jet')
ax2.axis('off')
ax2.set_title('U posterior')
ax3.imshow(qu_discrete, cmap='jet')
ax3.axis('off')
ax3.set_title('U posterior (discrete)')
plt.tight_layout()
plt.show()

### GPCA-net

TODO: add GPCA-net as encoder layer w/ trainable params W.

In [ ]:
# Load DESI normed image
data_path = '../data/integrated/'
desi_imgs = [tifffile.imread(os.path.join(data_path, 'desi', f))
             for f in sorted(os.listdir(os.path.join(data_path, 'desi')))
             if f[-3:] == 'tif']

# Load full-size u_prior from CyCIF image (graph-based heat diffusion)
u_priors = [tifffile.imread(os.path.join(data_path, 'zonation_prior', f))
            for f in sorted(os.listdir(os.path.join(data_path, 'zonation_prior')))
            if f[-3:] == 'tif']

nchans, ndimy, ndimx = desi_imgs[0].shape
yy, xx = np.meshgrid(np.arange(ndimy), np.arange(ndimx))
desi_coords = np.array([yy.flatten(), xx.flatten()]).T

gc.collect()

In [ ]:
# Get feature matrix from desi_image [C, X, Y] --> [X*Y, C]
idx = 10
desi_feature_mat = desi_imgs[idx].transpose(2,1,0).reshape(-1, nchans) 
desi_feature_mat = utils.norm_features(desi_feature_mat)

ims_loader = dataset.DESIGraphDataset(data_path = '../data/integrated/desi_debug/',
                                      prior_path = '../data/integrated/zonation_prior_debug/',
                                      k=4, n_subgraphs=8)
ims_data = ims_loader.load_graphs()
ims_train_dl = DataLoader(ims_data, shuffle=True)

In [ ]:
from importlib import reload
reload(utils)
reload(plot)
reload(model_train)
reload(model_eval)
reload(configs)
reload(dataset)
reload(gpca)
reload(vgae_sbm)

In [ ]:
lr = 1e-2
n_epochs = 100
device = torch.device('cuda')

train_configs = configs.set_train_configs(data_path=None, 
                                          n_epochs=n_epochs, 
                                          gamma=0.99,
                                          lr=lr)

model_configs = configs.set_model_configs(device=device,
                                          beta=0.5,
                                          c_in=nchans,
                                          c_hidden=8,
                                          c_latent=1,
                                          c0=5,
                                          alpha=0.5,
                                          pu_scale=0.5,
                                          dropout=0.01)


In [ ]:
model = vgae_sbm.SparseVGAE(encoder=gpca.GPCAEncoder(model_configs),
                            decoder=vgae_sbm.Decoder(model_configs),
                            beta=model_configs.beta)

optimizer = torch.optim.Adam(model.parameters(), lr=train_configs.lr)

In [ ]:
losses, nlls, l1s, ortho_losses, kls, signs = model_train.train(model, ims_train_dl, train_configs)

In [ ]:
fig, (ax1, ax2, ax3, ax4, ax5, ax6) = plt.subplots(6, 1, figsize=(10, 20))

ax1.plot(np.arange(n_epochs), losses)
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Total Loss')

ax2.plot(np.arange(n_epochs), nlls)
ax2.set_xlabel('Epochs')
ax2.set_ylabel('NLLs')

ax3.plot(np.arange(n_epochs), l1s)
ax3.set_xlabel('Epochs')
ax3.set_ylabel('L1 regularization')

ax4.plot(np.arange(n_epochs), ortho_losses)
ax4.set_xlabel('Epochs')
ax4.set_ylabel('Orthogonal regularization')

ax5.plot(np.arange(n_epochs), kls)
ax5.set_xlabel('Epochs')
ax5.set_ylabel('KLs')

ax6.plot(np.arange(n_epochs), signs)
ax6.set_xlabel('Epochs')
ax6.set_ylabel('Sign')

plt.tight_layout()
plt.show()

In [ ]:
# Graph reconstruction checks on full graph

idx = 10
desi_feature_mat = desi_imgs[idx].transpose(2, 1, 0).reshape(-1, nchans)
desi_feature_mat = utils.norm_features(desi_feature_mat)

G_desi = gen_graph.construct_graph(desi_coords, k=8, weighted=False)  
graph_data =  pyg_utils.from_networkx(G_desi)
edge_index, edge_weight = graph_data.edge_index, None
# A = nx.adjacency_matrix(G_desi).toarray()

model.eval()
with torch.no_grad():
    latent, recon = model_eval.eval(model, G_desi, desi_feature_mat)

qz = latent.qz_loc.detach().cpu().numpy()
qu = latent.qt.detach().cpu().numpy()
px = recon.px_loc.detach().cpu().numpy()
px_chan = px.reshape(ndimy, ndimx, -1).transpose(2,1,0)  # dim: [C, Y, X]
del graph_data
gc.collect()

In [ ]:
plt.figure(figsize=(4, 2), dpi=100)
plt.hist(latent.qt.flatten().squeeze(), edgecolor='white', bins=30)
plt.title('q(t) distribution')
plt.show()

plt.figure(figsize=(4, 2), dpi=100)
plt.hist(qz.flatten().squeeze(), edgecolor='white', bins=30)
plt.title('q(z) distribution')
plt.show()

plt.figure(figsize=(4, 2), dpi=100)
plt.hist(latent.qz.flatten().squeeze(), edgecolor='white', bins=30)
plt.title('q(z) * q(b) distribution')
plt.show()


In [ ]:
sample_idx = np.random.choice(np.arange(desi_feature_mat.shape[0]), 500, replace=False)

fig, ax = plt.subplots()
ax.scatter(desi_feature_mat[sample_idx, :].flatten(), px[sample_idx, :].flatten(), s=0.1)
ax.spines[['right', 'top']].set_visible(False)
ax.get_xaxis().tick_bottom()
ax.get_yaxis().tick_left()
ax.set_title('Feature matrix reconstruction')
plt.show()

del sample_idx
gc.collect()

In [ ]:
# Visuzlize spatial distribution of Node (n) ->module (z) assignments

def disp_module_spatial(z, height, ncol=4, 
                        panel_size=3, cmap='turbo'):
    assert z.shape[0] % height == 0,\
         "2D spatial plots should have int height & width"
    n_factors = z.shape[-1]  # dim: [N x factor]
    nrow = n_factors // ncol if n_factors % ncol == 0 else n_factors // ncol + 1
    width = z.shape[0] // height
    
    idx = 0
    fig, axes = plt.subplots(nrow, ncol, figsize=((panel_size+0.2)*ncol, panel_size*nrow), dpi=100)
    for r in range(nrow):
        for c in range(ncol):
            axes[r, c].axis('off')
            if idx < n_factors:
                axes[r, c].imshow(z[:, idx].reshape(height, width).T, cmap=cmap)
                axes[r, c].set_title('Factor {}'.format(idx))
            idx += 1
    plt.tight_layout()
    plt.show()

disp_module_spatial(qz, height=desi_imgs[idx].shape[-1],
                    panel_size=4, ncol=3)

In [ ]:
# Interprete metabolites (x) -> module (z) assignments (USE encoder weight instead!!!)
# module_weights = torch.softmax(model.encoder.x_to_zloc.weight, dim=-1).detach().cpu().numpy()
module_weights = model.encoder.x_to_zloc.weight.detach().cpu().numpy()
g = sns.clustermap(module_weights, method='ward', cmap='coolwarm', col_cluster=False)
g.ax_heatmap.axes.set_title('Metabolite module contribution', y=1.3, fontsize=20)
plt.show()

In [ ]:
plot.disp_network_embedding(desi_imgs[idx].mean(0), G_desi, embedding=qu.flatten(), alpha=0.5, 
                       node_size=5, edge_width=0.5, fontsize=64,
                       figsize=(15, 15), title='U posterior')

In [ ]:
# Create discretized zonation maps from posterior `u`
qu_2d = qu.reshape(-1, desi_imgs[idx].shape[-1]).T
qu_2d_discrete = utils.infer_zones(qu_2d, nbins=10)

plt.figure()
plt.imshow(qu_2d_discrete, cmap='turbo')
plt.colorbar()
plt.title('Discretised trajectory \n Hierarchical VGAE', fontsize=15)
plt.axis('off')
plt.show()

### VGAE

Load DESI normalized iamge & CyCIF prior computation

In [ ]:
# Load DESI normed image
data_path = '../data/integrated/'
desi_imgs = [tifffile.imread(os.path.join(data_path, 'desi', f))
             for f in sorted(os.listdir(os.path.join(data_path, 'desi')))
             if f[-3:] == 'tif']

# Load full-size u_prior from CyCIF image (graph-based heat diffusion)
u_priors = [tifffile.imread(os.path.join(data_path, 'zonation_prior', f))
            for f in sorted(os.listdir(os.path.join(data_path, 'zonation_prior')))
            if f[-3:] == 'tif']

nchans, ndimy, ndimx = desi_imgs[0].shape
yy, xx = np.meshgrid(np.arange(ndimy), np.arange(ndimx))
desi_coords = np.array([yy.flatten(), xx.flatten()]).T

gc.collect()

# Plot k-NN graph
# G_desi = gen_graph.construct_graph(desi_coords, k=8, r=10)  
# plot.disp_network(desi_img[7], G_desi, figsize=(12, 12), node_size=0.8, edge_width=0.5)

Train on subgraphs:

In [ ]:
# Debug: train on example 2D slice

# Get feature matrix from desi_image [C, X, Y] --> [X*Y, C]
# idx = 10
# desi_feature_mat = desi_imgs[idx].transpose(2,1,0).reshape(-1, nchans) 
# desi_feature_mat = utils.norm_features(desi_feature_mat)

# ims_loader = dataset.DESIGraphDataset(data_path = '../data/integrated/desi_debug/',
#                                       prior_path = '../data/integrated/zonation_prior_debug/',
#                                       k=4, n_subgraphs=8)
# ims_data = ims_loader.load_graphs()
# ims_train_dl = DataLoader(ims_data, shuffle=True)

# Train on whole 3D slices
# ims_loader = dataset.DESIGraphDataset(data_path='../data/integrated/desi/',
#                                       prior_path='../data/integrated/zonation_prior/',
#                                       k=4, n_subgraphs=8)
# ims_data = ims_loader.load_graphs()

# Saving subgraphs
# subgraphs = []
# for x in ims_data:
#     subgraph = pyg_utils.to_networkx(x, node_attrs=['pos'], to_undirected=True)
#     subgraphs.append(subgraph)
# del x, subgraph

ims_train, ims_test = torch.utils.data.random_split(ims_data, [0.5, 0.5],
                                                    generator=torch.Generator().manual_seed(42))
# Split training & validation set
ims_train_dl = DataLoader(ims_train, shuffle=True)
ims_test_dl = DataLoader(ims_test)

Model training:

In [ ]:
# DEBUG model
from importlib import reload
reload(baseline)
reload(vgae_sbm)
reload(utils)
reload(plot)
reload(model_train)
reload(model_eval)
reload(configs)
reload(dataset)
# del model, optimizer

In [ ]:
lr = 1e-2
n_epochs = 200
device = torch.device('cuda')

train_configs = configs.set_train_configs(data_path=None, 
                                          n_epochs=n_epochs, 
                                          gamma=0.99,
                                          lr=lr)

model_configs = configs.set_model_configs(device=device,
                                          beta=0.5,
                                          c_in=nchans,
                                          c_hidden=8,
                                          c_latent=1,
                                          c0=5.0,
                                          px_scale=1.0,
                                          dropout=0.05)

model = vgae_sbm.SparseVGAE(encoder=vgae_sbm.GCNEncoder(model_configs),
                            decoder=vgae_sbm.Decoder(model_configs),
                            beta=model_configs.beta)

optimizer = torch.optim.Adam(model.parameters(), lr=train_configs.lr)

In [ ]:
losses, nlls, l1s, ortho_losses, kls, signs = model_train.train(model, ims_train_dl, train_configs)

In [ ]:
fig, (ax1, ax2, ax3, ax4, ax5, ax6) = plt.subplots(6, 1, figsize=(10, 20))

ax1.plot(np.arange(n_epochs), losses)
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Total Loss')

ax2.plot(np.arange(n_epochs), nlls)
ax2.set_xlabel('Epochs')
ax2.set_ylabel('NLLs')

ax3.plot(np.arange(n_epochs), l1s)
ax3.set_xlabel('Epochs')
ax3.set_ylabel('L1 regularization')

ax4.plot(np.arange(n_epochs), ortho_losses)
ax4.set_xlabel('Epochs')
ax4.set_ylabel('Orthogonal regularization')

ax5.plot(np.arange(n_epochs), kls)
ax5.set_xlabel('Epochs')
ax5.set_ylabel('KLs')

ax6.plot(np.arange(n_epochs), signs)
ax6.set_xlabel('Epochs')
ax6.set_ylabel('Sign')

plt.tight_layout()
plt.show()

In [ ]:
torch.save(model, '../data/desi/res/model_int.pt')

In [ ]:
# Graph reconstruction checks on full grapha

idx = 1
desi_feature_mat = desi_imgs[idx].transpose(2, 1, 0).reshape(-1, nchans)
desi_feature_mat = utils.norm_features(desi_feature_mat)

G_desi = gen_graph.construct_graph(desi_coords, k=8, weighted=False)  
graph_data =  pyg_utils.from_networkx(G_desi)
edge_index, edge_weight = graph_data.edge_index, None
# A = nx.adjacency_matrix(G_desi).toarray()

model.eval()
with torch.no_grad():
    latent, recon = model_eval.eval(model, G_desi, desi_feature_mat)

qz = latent.qz.detach().cpu().numpy()
qu = latent.qu.detach().cpu().numpy()
px = recon.px_loc.detach().cpu().numpy()
px_chan = px.reshape(ndimy, ndimx, -1).transpose(2,1,0)  # dim: [C, Y, X]
del graph_data
gc.collect()

Predictive checks:

In [ ]:
# plot distributions of u, z 

plt.figure(figsize=(4, 2), dpi=100)
plt.hist(qu.flatten().squeeze(), edgecolor='white', bins=30)
plt.title('q(u) distribution')
plt.show()

plt.figure(figsize=(4, 2), dpi=100)
plt.hist(qz.flatten().squeeze(), edgecolor='white', bins=30)
plt.title('q(z) distribution')
plt.show()


In [ ]:
sample_idx = np.random.choice(np.arange(desi_feature_mat.shape[0]), 500, replace=False)

fig, ax = plt.subplots()
ax.scatter(desi_feature_mat[sample_idx, :].flatten(), px[sample_idx, :].flatten(), s=0.1)
ax.spines[['right', 'top']].set_visible(False)
ax.get_xaxis().tick_bottom()
ax.get_yaxis().tick_left()
ax.set_title('Feature matrix reconstruction')
plt.show()

del sample_idx
gc.collect()

In [ ]:
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
# ax1.imshow(A[:1024, :1024])
# ax1.axis('off')
# ax2.imshow(A_hat[:1024, :1024])
# ax2.axis('off')
# plt.tight_layout()
# plt.show()

In [ ]:
# Soft membership assignment (latent.qb):
g = sns.clustermap(latent.qb.detach().cpu().numpy(), col_cluster=False)
g.ax_heatmap.axes.set_title('Node cluster assignment (B)', fontsize=20)

In [ ]:
g = sns.clustermap(qz, col_cluster=False)
g.ax_heatmap.axes.set_title('Node cluster strength (Z)', fontsize=20)
# sns.clustermap(z)`1

In [ ]:
# Visuzlize spatial distribution of Node (n) ->module (z) assignments

def disp_module_spatial(z, height, ncol=4, 
                        panel_size=3, cmap='turbo'):
    assert z.shape[0] % height == 0,\
         "2D spatial plots should have int height & width"
    n_factors = z.shape[-1]  # dim: [N x factor]
    nrow = n_factors // ncol if n_factors % ncol == 0 else n_factors // ncol + 1
    width = z.shape[0] // height
    
    idx = 0
    fig, axes = plt.subplots(nrow, ncol, figsize=((panel_size+0.2)*ncol, panel_size*nrow), dpi=100)
    for r in range(nrow):
        for c in range(ncol):
            axes[r, c].axis('off')
            if idx < n_factors:
                axes[r, c].imshow(z[:, idx].reshape(height, width).T, cmap=cmap)
                axes[r, c].set_title('Factor {}'.format(idx))
            idx += 1
    plt.tight_layout()
    plt.show()

disp_module_spatial(qz, height=desi_imgs[idx].shape[-1],
                    panel_size=4, ncol=3)

In [ ]:
# Correlations btw Zs
fig, ax = plt.subplots(figsize=(6, 6), dpi=200)
sns.heatmap(np.corrcoef(qz.T), vmin=-1, vmax=1, cmap='coolwarm', ax=ax, square=True)
fig.suptitle('Correlations btw z-factors')
plt.show()


In [ ]:
# module_weights = torch.softmax(model.decoder.z_to_xloc.lin_l.weight, dim=-1).detach().cpu().numpy()
module_weights = model.decoder.z_to_xloc.lin_l.weight.detach().cpu().numpy()

g = sns.clustermap(module_weights, method='ward', cmap='coolwarm', col_cluster=False)
g.ax_heatmap.axes.set_title('Metabolite module contribution', y=1.3, fontsize=20)
plt.show()

In [ ]:
# Visualize UMAPs of z & x
adata_z = sc.AnnData(qz)
for i in range(z.shape[1]):
    label = 'z'+str(i)
    adata_z.obs[label] = z_norm[:, i]

sc.pp.neighbors(adata_z)
sc.tl.umap(adata_z)

In [ ]:
sc.pl.umap(adata_z, color=adata_z.obs.columns)

In [ ]:
adata_x = sc.AnnData(desi_feature_mat)

# Label DESI features w/ latents?
for i in range(z.shape[1]):
    label = 'z'+str(i)
    adata_x.obs[label] = z[:, i]

sc.pp.pca(adata_x)
sc.pp.neighbors(adata_x)
sc.tl.umap(adata_x)

In [ ]:
# sc.pl.pca_variance_ratio(adata_x)
sc.pl.umap(adata_x, color=adata_x.obs.columns)
# del adata_z, adata_x

In [ ]:
# Visualize prior & posterior `u`:

# Note: coords saved in [x,y]-index in graph, but in [i,j]-index in image!!:
# transpose 2D image before flatten
# plot.disp_network_embedding(desi_imgs[idx].mean(0), G_desi, embedding=u_priors[idx].T.flatten(), alpha=0.5, 
#                        node_size=5, edge_width=0.5, fontsize=64,
#                        figsize=(15, 15), title='U prior (CyCIF)')

plot.disp_network_embedding(desi_imgs[idx].mean(0), G_desi, embedding=qu.flatten(), alpha=0.5, 
                       node_size=5, edge_width=0.5, fontsize=64,
                       figsize=(15, 15), title='U posterior')

In [ ]:
# Create discretized zonation maps from posterior `u`
qu_2d = qu.reshape(-1, desi_imgs[idx].shape[-1]).T
qu_2d_discrete = utils.infer_zones(qu_2d, nbins=10)

plt.figure()
plt.imshow(qu_2d_discrete, cmap='turbo')
plt.colorbar()
plt.title('Discretised trajectory \n Hierarchical VGAE', fontsize=15)
plt.axis('off')
plt.show()

In [ ]:
# Save U posterior
np.save('../data/desi/res/u_posterior.npy', qu)

### Result Analysis

#### 2D 

Visualize DESI gradients sorted on a sample posterior `U`:

In [ ]:
idx = 1

# Load DESI m/z labels:
ifile = open('../data/integrated/desi/DESI_slice_07.ome.tif', 'rb')
ion_labels = IO.load_ome_labels(ifile, '../data/integrated/desi/DESI_slice_07.ome.tif')
ifile.close()

In [ ]:
# Visualize DESI features sorted by prior temperature
desi_feature_mat_sorted = desi_feature_mat.reshape(-1, desi_feature_mat.shape[-1])  # Coord x Feature
desi_feature_mat_sorted = desi_feature_mat_sorted[np.argsort(u_priors[idx].T.flatten())]

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Prior Temperature', nbins=10)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Prior Temperature', nbins=20)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                        ion_labels, 
                        title='Prior Temperature', nbins=100)


In [ ]:
# Visualize DESI features sorted by POSTERIOR temperature
desi_feature_mat_sorted = desi_feature_mat.reshape(-1, desi_feature_mat.shape[-1])  # Coord x Feature
desi_feature_mat_sorted = desi_feature_mat_sorted[np.argsort(qu.squeeze())] # sort by posterior temp

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels,
                         title='Posterior Temperature', nbins=10)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Posterior Temperature', nbins=20)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Posterior Temperature', nbins=100)

In [ ]:
from util import utils
reload(utils)

In [ ]:
nbins = 100

x_means, x_stds, grad_indices = utils.sort_binned_features(desi_feature_mat_sorted, nbins=nbins)

ion_labels_sorted = np.array(ion_labels)[grad_indices]
desi_img_sorted = desi_imgs[idx][[grad_indices], ...]
px_sorted = px[:, [grad_indices]]

desi_binned_grads_df = pd.DataFrame(x_means,  
                                    index=ion_labels_sorted,
                                    columns=['bin' + str(i) for i in range(nbins)])

display(desi_binned_grads_df.head())
# desi_binned_grads_df.to_csv('../data/desi/res/metabolite_binned_gradients.csv', index=True)
del nbins

In [ ]:
plot.disp_desi_gradients(desi_binned_grads_df.values.T, 
                         ion_labels_sorted, cluster_ions=False,
                         title='Posterior Temperature', nbins=100)

In [ ]:
# Smoothed gradient plots of indiv. features:
from scipy.stats import linregress

nbins = 100
desi_gradient_ts = []  # Fitted slope of the features along PV -> CV trajectory

for feature_mean, feature_std, label in zip(desi_binned_grads, desi_binned_stds, ion_labels_sorted):
    # plot.disp_desi_gradient(feature_mean, feature_std, title=label)
    ts = linregress(np.linspace(-1, 1, nbins), feature_mean)
    desi_gradient_ts.append(ts)

del label, feature_mean, feature_std

In [ ]:
# Summarize expression gradienst along PV->CV trajectory
desi_gradient_slopes = [ts.slope for ts in desi_gradient_ts]
desi_gradient_pvals = [ts.pvalue for ts in desi_gradient_ts]
desi_gradient_categories = []
for slope, pval in zip(desi_gradient_slopes, desi_gradient_pvals):
    if pval >= 0.05:
        desi_gradient_categories.append('Uncertain')
    elif slope > 0:
        desi_gradient_categories.append('+')
    else:
        desi_gradient_categories.append('-')


In [ ]:
valid_slopes = [slope 
                for (slope, p) in zip(desi_gradient_slopes, desi_gradient_pvals)
                if p < 0.05]

plt.figure(figsize=(5, 2))
plt.hist(valid_slopes, edgecolor='white', bins=10)
plt.xlabel(r'Fitted slope $\beta$')
plt.suptitle('Metabolite gradients along PV->CV trajectory')
plt.show()

del valid_slopes

In [ ]:
# summarize metabolite gradient as slopes
gradient_df = pd.DataFrame({
    'Label': ion_labels_sorted,
    'Slope': desi_gradient_slopes,
    'Category': desi_gradient_categories,
    'p-val': desi_gradient_pvals
})

gradient_df.set_index('Label', inplace=True)
# gradient_df.to_csv('../data/desi/res/metabolite_gradients.csv', index=True)
gradient_df.head()

Check annotated metabolites:

SEAM:
- FA: `m/z 255.22, 279.22, 281.23`
- Kupffer cell metabolites: `m/z 134.04, 180.89` (Adanine: `m/z 134.04`)
- Glutamine: `m/z 145.04`

In [ ]:
annot_labels = [
    '865.49527 m/z ± 50 mDa',
    '267.08191 m/z ± 50 mDa',
    '295.22855 m/z ± 50 mDa',
    '261.14629 m/z ± 50 mDa', 
    '253.21007 m/z ± 25.16 mDa', 
    '301.22061 m/z ± 50 mDa',
    '303.24218 m/z ± 50 mDa',
    '346.06614 m/z ± 50 mDa',
    '609.52634 m/z ± 50 mDa',
    '699.51653 m/z ± 50 mDa'
]


In [ ]:
annot_indices = []
for i, (feature_mean, feature_std, label) in enumerate(zip(desi_binned_grads, 
                                                           desi_binned_stds, 
                                                           ion_labels_sorted)):
    if label in annot_labels:
        annot_indices.append(i)

        # Trajectory plot
        plot.disp_desi_gradient(feature_mean, feature_std, title=label)

        # feature image
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), dpi=100)
        ax1.imshow(desi_img_sorted[i], cmap='magma')
        ax1.set_title(label + '\n(raw)')
        ax2.imshow(px_sorted[:, i].reshape(ndimy, ndimx).T, cmap='magma')
        ax2.set_title(label + '\n(reconst)')
        plt.tight_layout()
        plt.show()

del label, feature_mean, feature_std

In [ ]:
# Learnt metabolites->module assignments
W_xz_sorted = module_weights[grad_indices]
g = sns.clustermap(W_xz_sorted[annot_indices], 
                   method='ward',
                   yticklabels=ion_labels_sorted[annot_indices],
                   xticklabels=['Module_' + str(i) for i in range(qz.shape[1])],
                   cmap='coolwarm', square=True, col_cluster=False,
                   figsize=(6, 6))

plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45)
g.ax_heatmap.axes.set_title('Annotated metabolites & Module weights', y=1.3, fontsize=15)
plt.show()

Example associatations btw soft-assignments & Module weights:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(qz[:, 2].reshape(-1, desi_imgs[idx].shape[-1]).T, cmap='turbo')
ax1.set_title('Module 1')
ax1.axis('off')

ax2.imshow(desi_img_sorted[annot_indices[1]], cmap='magma')  # Metabolite 267
ax2.set_title('Representative PV metabolites')
ax2.axis('off')
plt.show()

In [ ]:
# Learnt metabolites->module assignments
W_xz_sorted = module_weights[np.argsort(argmax_grad)]

g = sns.clustermap(np.corrcoef(W_xz_sorted[annot_indices]),
                   method='ward',
                   xticklabels=[],
                   yticklabels=ion_labels_sorted[annot_indices],
                   figsize=(6, 4),
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of module assignments\n btw annotated DESI features', y=1.3, fontsize=20)
plt.show()

In [ ]:
# Binned-expression correlations
g = sns.clustermap(desi_binned_grads_df.loc[annot_labels].T.corr(), 
                   method='ward',
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n btw annotated DESI features', y=1.3, fontsize=20)
plt.show()

In [ ]:
# Raw-expresssion correlations 
g = sns.clustermap(np.corrcoef(desi_feature_mat_sorted[:, annot_indices].T),
                   method='ward',
                   xticklabels=ion_labels_sorted[annot_indices],
                   yticklabels=ion_labels_sorted[annot_indices],
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of raw-expressions\n btw annotated DESI features', y=1.3, fontsize=20)
plt.show()

**TODO: spatial correlation metrics (e.g. Moran's I)**

#### 3D integration

- Cluster metabolites with similar trajectory
- Check 3D patterning for annotated metabolites (whether necessary to perform any smoothing to account for noises)

Infer latents across slices:

In [ ]:
# Load data:
# ...

In [ ]:
# Load model:
# ...

In [ ]:
del desi_features_3d

In [ ]:
# # Common graph structure for each L-slice
G_desi = gen_graph.construct_graph(desi_coords, k=4)  

b_list = []  # Binary spatial cluster assignment
z_list = []  # Spatial node cluster strength assignment
u_list = []  # Trajectory assignment 
x_3d = []
px_3d = []
module_weights_list = []

with torch.no_grad():
    for i, desi_img in enumerate(desi_imgs):
        x = desi_img.transpose(2, 1, 0).reshape(-1, nchans)
        x = utils.norm_features(x)
        x_3d.append(x)
        
        latent, recon = model_eval.eval(model, G_desi, x)
        b = latent.qb.detach().cpu().numpy()
        z = latent.qz.detach().cpu().numpy()
        u = latent.qu.detach().cpu().numpy()
        w = model.decoder.z_to_xloc.lin_l.weight.detach().cpu().numpy()
        px = recon.px_loc.detach().cpu().numpy()
        
        b_list.append(b)
        z_list.append(z)
        u_list.append(u)
        module_weights_list.append(w)
        px_3d.append(px)
        
del desi_img, latent, recon, b, z, u, w, x

In [ ]:
u_post_list_2d = [u.reshape(ndimy, ndimx).T for u in u_list]
plot.disp_chans(u_post_list_2d, cmap='turbo', title=r'Zonation posterior ($u$)')
#del u_post_list_2d

In [ ]:
u_post_discrete_2d = [utils.infer_zones(u_post, nbins=10) for u_post in u_post_list_2d]
plot.disp_chans(u_post_discrete_2d, cmap='coolwarm', title=r'Discretized zonation posterior ($u$)')

In [ ]:
tifffile.imwrite('../data/desi/res/u_posteriors.tif', np.array(u_post_list_2d))
tifffile.imwrite('../data/desi/res/u_posteriors_discretized.tif', np.array(u_post_discrete_2d))
tifffile.imwrite('../data/desi/res/z_3d.tif', np.array(z_list))

Check 3D trajectory of representative metabolites:

In [ ]:
x_3d = np.array(x_3d)  # dim: [L, Y*X, C]
x_3d_unsorted = x_3d.transpose(2, 0, 1).reshape(nchans, -1)  # dim: [C, L*Y*X] unsorted
x_3d_sorted = np.zeros_like(x_3d_unsorted)  # expressions sorted by inferred gradient values, dim: [C, L*Y*X]

indices = np.argsort(np.array(u_list).squeeze().flatten())
for i, chan in enumerate(x_3d_unsorted):
    x_3d_sorted[i] = chan[indices]

del x_3d_unsorted, indices, chan

In [ ]:
px_3d = np.array(px_3d)  # dim: [L, Y*X, C]
px_3d_unsorted = px_3d.transpose(2, 0, 1).reshape(nchans, -1)  # dim: [C, L*Y*X] unsorted
px_3d_sorted = np.zeros_like(px_3d_unsorted)  # expressions sorted by inferred gradient values, dim: [C, L*Y*X]

indices = np.argsort(np.array(u_list).squeeze().flatten())
for i, chan in enumerate(px_3d_unsorted):
    px_3d_sorted[i] = chan[indices]

del px_3d_unsorted, indices, chan

In [ ]:
# Visualize binned trajectory:
plot.disp_desi_gradients(x_3d_sorted.T, 
                         ion_labels, 
                         title='Posterior gradient (3D)', nbins=100)

In [ ]:
# Visualize binned trajectory:
plot.disp_desi_gradients(px_3d_sorted.T, 
                         ion_labels, 
                         title='Posterior denoised gradient (3D)', nbins=100)

In [ ]:
# Full annotation labels - separation btw/ categories

FA_labels = [
    '193.11679 m/z ± 50 mDa',
    '195.14353 m/z ± 22.09 mDa',
    '253.21007 m/z ± 25.16 mDa',
    '255.22685 m/z ± 50 mDa',
    '277.22812 m/z ± 39.49 mDa',
    '279.2325 m/z ± 50 mDa',
    '281.2441 m/z ± 39.78 mDa',
    '283.26292 m/z ± 50 mDa',
    '287.21503 m/z ± 50 mDa',
    '289.17446 m/z ± 50 mDa',
    '291.19452 m/z ± 50 mDa',
    '293.2216 m/z ± 50 mDa',
    '295.22855 m/z ± 50 mDa',
    '297.05155 m/z ± 50 mDa',
    '301.22061 m/z ± 50 mDa',
    '303.24218 m/z ± 50 mDa',
    '305.25545 m/z ± 50 mDa',
    '307.27538 m/z ± 50 mDa',
    '311.22364 m/z ± 41.84 mDa',
    '311.30732 m/z ± 41.84 mDa',
    '313.2352 m/z ± 50 mDa',
    '315.1971 m/z ± 28.07 mDa',
    '315.25325 m/z ± 28.07 mDa',
    '317.22145 m/z ± 50 mDa',
    '319.22402 m/z ± 50 mDa',
    '325.21252 m/z ± 50 mDa',
    '327.24012 m/z ± 50 mDa',
    '329.24533 m/z ± 50 mDa',
    '331.25667 m/z ± 50 mDa',
    '333.2164 m/z ± 50 mDa',
    '333.27413 m/z ± 50 mDa',
    '335.23981 m/z ± 50 mDa',
    '339.21764 m/z ± 43.68 mDa',
    '345.24306 m/z ± 50 mDa',
    '349.22053 m/z ± 50 mDa',
    '351.23262 m/z ± 50 mDa',
]

ST_labels = [
    '309.21855 m/z ± 27.8 mDa',
    '343.21893 m/z ± 50 mDa',
    '355.27415 m/z ± 50 mDa',
    '417.34599 m/z ± 50 mDa',
    '448.32793 m/z ± 50 mDa',
    '477.37418 m/z ± 50 mDa',
    '493.32723 m/z ± 50 mDa',
    '498.29133 m/z ± 50 mDa',
]

PE_labels = [
    '714.52524 m/z ± 50 mDa',  # LPE!
    '738.51815 m/z ± 50 mDa',
    '740.53903 m/z ± 50 mDa',
    '742.51959 m/z ± 50 mDa',
    '762.51427 m/z ± 50 mDa',
    '764.52398 m/z ± 50 mDa',
    '766.53633 m/z ± 50 mDa',
]

PA_labels = [
    '723.47084 m/z ± 50 mDa',
    '817.52389 m/z ± 50 mDa',
    '819.51426 m/z ± 50 mDa'
]

PI_labels = [
    '861.54597 m/z ± 50 mDa',
    '863.58919 m/z ± 50 mDa',
    '885.56543 m/z ± 50 mDa'
]


In [ ]:
# Compute binned trajectory
nbins = 100
x_means, x_stds, grad_indices = utils.sort_binned_features(x_3d_sorted.T, nbins)

ion_labels_sorted = np.array(ion_labels)[grad_indices]
desi_imgs_sorted = [img[grad_indices, ...] for img in desi_imgs]

desi_grads_df = pd.DataFrame(x_means,  
                             index=ion_labels_sorted,
                             columns=['bin' + str(i) for i in range(nbins)])

# display(desi_grads_df.head())
# desi_grads_df.to_csv('../data/desi/res/metabolite_binned_gradients_3d.csv', index=True)
del nbins

In [ ]:
for i, (x_mean, x_std, label) in enumerate(zip(x_means, x_stds, ion_labels_sorted)):
    if label in FA_labels:
        # Trajectory plot
        plot.disp_desi_gradient(x_mean, x_std, title=label + ' (3D)')
del x_mean, x_std, label

In [ ]:
for i, (x_mean, x_std, label) in enumerate(zip(x_means, x_stds, ion_labels_sorted)):
    if label in ST_labels:
        # Trajectory plot
        plot.disp_desi_gradient(x_mean, x_std, title=label + ' (3D)')
        plt.figure(figsize=(6, 6), dpi=80)
        plt.imshow(desi_imgs_sorted[10][i], cmap='magma')
        plt.title(label)
        plt.axis('off')
        plt.show()
del x_mean, x_std, label

In [ ]:
for i, (x_mean, x_std, label) in enumerate(zip(x_means, x_stds, ion_labels_sorted)):
    if label in PA_labels:
        # Trajectory plot
        plot.disp_desi_gradient(x_mean, x_std, title=label + ' (3D)')

        plt.figure(figsize=(6, 6), dpi=80)
        plt.imshow(desi_imgs_sorted[10][i], cmap='magma')
        plt.title(label)
        plt.axis('off')
        plt.show()
        
del x_mean, x_std, label

In [ ]:
for i, (x_mean, x_std, label) in enumerate(zip(x_means, x_stds, ion_labels_sorted)):
    if label in PE_labels:
        # Trajectory plot
        plot.disp_desi_gradient(x_mean, x_std, title=label + ' (3D)')

        plt.figure(figsize=(6, 6), dpi=80)
        plt.imshow(desi_imgs_sorted[10][i], cmap='magma')
        plt.title(label)
        plt.axis('off')
        plt.show()
        
del x_mean, x_std, label

In [ ]:
for i, (x_mean, x_std, label) in enumerate(zip(x_means, x_stds, ion_labels_sorted)):
    if label in PI_labels:
        # Trajectory plot
        plot.disp_desi_gradient(x_mean, x_std, title=label + ' (3D)')

        plt.figure(figsize=(6, 6), dpi=80)
        plt.imshow(desi_imgs_sorted[10][i], cmap='magma')
        plt.title('label')
        plt.axis('off')
        plt.show()
        
del x_mean, x_std, label

In [ ]:
# Try posterior denoised expression p(x|z)
nbins = 100
px_means, px_stds, grad_indices = utils.sort_binned_features(px_3d_sorted.T, nbins)
ion_labels_sorted = np.array(ion_labels)[grad_indices]

del nbins

In [ ]:
for i, (px_mean, px_std, label) in enumerate(zip(px_means, px_stds, ion_labels_sorted)):
    if label in FA_labels:
        # Trajectory plot
        plot.disp_desi_gradient(px_mean, px_std, title=label + ' (3D)')
del px_mean, px_std, label

In [ ]:
for i, (px_mean, px_std, label) in enumerate(zip(px_means, px_stds, ion_labels_sorted)):
    if label in ST_labels:
        # Trajectory plot
        plot.disp_desi_gradient(px_mean, px_std, title=label + ' (3D)')
del px_mean, px_std, label

In [ ]:
for i, (px_mean, px_std, label) in enumerate(zip(px_means, px_stds, ion_labels_sorted)):
    if label in PA_labels:
        # Trajectory plot
        plot.disp_desi_gradient(px_mean, px_std, title=label + ' (3D)')
del px_mean, px_std, label

In [ ]:
for i, (px_mean, px_std, label) in enumerate(zip(px_means, px_stds, ion_labels_sorted)):
    if label in PE_labels:
        # Trajectory plot
        plot.disp_desi_gradient(px_mean, px_std, title=label + ' (3D)')
del px_mean, px_std, label

In [ ]:
for i, (px_mean, px_std, label) in enumerate(zip(px_means, px_stds, ion_labels_sorted)):
    if label in PI_labels:
        # Trajectory plot
        plot.disp_desi_gradient(px_mean, px_std, title=label + ' (3D)')
del px_mean, px_std, label

SEAM m/z annotation:
- FA: `m/z 227.18, 251.20, 253.20, 255.22, 279.22, 281.23, 283.35`
- Kupffer cell metabolites: `m/z 134.04, 180.89` (Adanine: `m/z 134.04`)
- Glutamine: `m/z 145.04`
- Guanine?: `m/z 150.99`
- Phosphocholine-CH3?: `m/z 168.05`

In [ ]:
# Annotation of known m/z reported from other papers?? 
for i, label in enumerate(ion_labels_sorted):
    if '283' in label:
        print(i, label)

In [ ]:
ion_idx = 179
plot.disp_desi_gradient(desi_grads[ion_idx], desi_stds[ion_idx], title=ion_labels_sorted[ion_idx] + ' (3D)')
del ion_idx

In [ ]:
# Module weight correlation
module_weights = np.array(module_weights_list).mean(0)
module_weights_df = pd.DataFrame(module_weights,
                                 index=ion_labels,
                                 columns=['Factor_{0}'.format(i) for i in range(module_weights.shape[1])])

In [ ]:
g = sns.clustermap(module_weights_df, method='ward', cmap='coolwarm', col_cluster=False)
g.ax_heatmap.axes.set_title('Metabolite module contribution', y=1.3, fontsize=20)
plt.show()

In [ ]:
?disp_module_spatial

In [ ]:
for qz in z_list:
    disp_module_spatial(qz, height=ndimy)

In [ ]:
g = sns.clustermap(module_weights_df.loc[annot_labels].T.corr(),
                   method='ward',
                   vmin=-1, 
                   vmax=1,
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of Module weights\n Highlighted ions', y=1.3, fontsize=20)
plt.show()

g = sns.clustermap(module_weights_df.loc[FA_labels].T.corr(),
                   method='ward',
                   vmin=-1, 
                   vmax=1,
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of Module weights\n (FA)', y=1.3, fontsize=20)
plt.show()

g = sns.clustermap(module_weights_df.loc[ST_labels].T.corr(),
                   method='ward',
                   vmin=-1, 
                   vmax=1,
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of Module weights\n (ST)', y=1.3, fontsize=20)

g = sns.clustermap(module_weights_df.loc[PE_labels].T.corr(),
                   method='ward',
                   vmin=-1, 
                   vmax=1,
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of Module weights\n (PE)', y=1.3, fontsize=20)
plt.show()

In [ ]:
# Binned-expression correlations
g = sns.clustermap(desi_grads_df.loc[FA_labels].T.corr(),
                   vmin=-1, 
                   vmax=1,
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n (FA)', y=1.3, fontsize=20)
plt.show()

g = sns.clustermap(desi_grads_df.loc[ST_labels].T.corr(),
                   vmin=-1, 
                   vmax=1,
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n (ST)', y=1.3, fontsize=20)
plt.show()

g = sns.clustermap(desi_grads_df.loc[PA_labels].T.corr(),
                   vmin=-1, 
                   vmax=1,
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n (PA)', y=1.3, fontsize=20)
plt.show()

g = sns.clustermap(desi_grads_df.loc[PE_labels].T.corr(),
                   vmin=-1, 
                   vmax=1,
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n (PE)', y=1.3, fontsize=20)
plt.show()

In [ ]:
# # Raw-expresssion correlations 
# g = sns.clustermap(np.corrcoef(x_3d_sorted[annot_indices, :]),
#                     xticklabels=ion_labels_sorted[annot_indices],
#                     yticklabels=ion_labels_sorted[annot_indices],
#                     cmap='coolwarm', square=True)
# g.ax_heatmap.axes.set_title('Correlations of raw-expressions\n btw annotated DESI features', y=1.3, fontsize=20)
# plt.show()

Group modules by (1). Module weight assignment; (2).binned trajectories via hierarchical clustering:

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

In [ ]:
# Cluster by Module weight assignment `W_xz`
linked_W_xz = linkage(module_weights_df, method='ward')

fig, ax = plt.subplots(figsize=(8, 20), dpi=500)
dendrogram(linked_W_xz, labels=ion_labels, color_threshold=3, 
           orientation='left', distance_sort='descending',
           show_leaf_counts=True, ax=ax)
plt.show()
fig.savefig('../data/desi/res/metabolite_module_W_xz.pdf', bbox_inches='tight', dpi=300)

In [ ]:
# Louvain cluster by trajectory `u`

# adata_mz_grads = sc.AnnData(desi_grads_df)  # [C, #bins]
# sc.pp.neighbors(adata_mz_grads, n_neighbors=10)
# sc.tl.louvain(adata_mz_grads)
# sc.tl.umap(adata_mz_grads)
# sc.pl.umap(adata_mz_grads, color='louvain', s=30)

In [ ]:
linked_grads = linkage(desi_grads_df, method='ward')

fig, ax = plt.subplots(figsize=(6, 18), dpi=300)
dendrogram(linked_grads, labels=desi_grads_df.index, color_threshold=3, 
           orientation='left', distance_sort='descending',
           show_leaf_counts=True, ax=ax)
plt.show()
fig.savefig('../data/desi/res/metabolite_module_gradients.pdf', bbox_inches='tight', dpi=300)

In [ ]:
# Tree-cut for clusters
max_d = 3
clusters = fcluster(linked_grads, max_d, criterion='distance')
desi_grads_df.insert(0, 'Cluster', clusters)
display(desi_grads_df.head())
desi_grads_df.to_csv('../data/desi/res/mz_modules_3d.csv', index=True)

### Benchmarks
Baseline methods on full 3D data

#### GASTON

In [ ]:
import gaston
from gaston import process_NN_output, dp_related, cluster_plotting

In [ ]:
gaston_model, A, S = process_NN_output.process_files('../data/desi/res/GASTON_output/')
# S = np.flip(S, axis=1)

In [ ]:
n_layers = 9
u_gaston, gaston_labels = dp_related.get_isodepth_labels(gaston_model, A, S, n_layers)
u_gaston = u_gaston.reshape(-1, desi_img.shape[-1])  # Convert to dim=[i, j]
u_gaston = -u_gaston  # tmp: Convert to relatively correct PV->CV trajectory

In [ ]:
plt.figure()
plt.imshow(u_gaston, cmap='turbo')
plt.colorbar()
plt.axis('off')
plt.title('GASTON isodepth', fontsize=15)
plt.show()

In [ ]:
# Visualize DESI features sorted by GASTON trajectory
desi_feature_mat_gaston = desi_feature_mat.reshape(-1, desi_feature_mat.shape[-1])  # Coord x Feature
desi_feature_mat_gaston = desi_feature_mat_gaston[np.argsort(u_gaston.T.flatten())]

plot.disp_desi_gradients(desi_feature_mat_gaston, 
                         ion_labels, 
                         title='GASTON isodepth', nbins=10)

plot.disp_desi_gradients(desi_feature_mat_gaston, 
                         ion_labels, 
                         title='GASTON isodepth', nbins=20)

plot.disp_desi_gradients(desi_feature_mat_gaston, 
                         ion_labels, 
                         title='GASTON isodepth', nbins=100)


In [ ]:
# Plot example metabolites along GASTON isodepth

nbins = 100
# Feature x nbins
desi_gaston_grads, desi_gaston_stds = utils.get_binned_features(desi_feature_mat_gaston, nbins=nbins) 
desi_gaston_grads, desi_gaston_stds = desi_gaston_grads.T, desi_gaston_stds.T

# Sort metabolites based on argmax values along the PV->CV binned trajectory
argmax_grad = desi_gaston_grads.argmax(1)

gaston_ion_labels_sorted = np.array(ion_labels)[np.argsort(argmax_grad)]
gaston_desi_img_sorted = desi_img[np.argsort(argmax_grad), ...]
desi_gaston_grads = desi_gaston_grads[np.argsort(argmax_grad), :]
desi_gaston_stds = desi_gaston_stds[np.argsort(argmax_grad), :]

desi_gaston_grads_df = pd.DataFrame(desi_gaston_grads,  
                                    index=gaston_ion_labels_sorted,
                                    columns=['bin' + str(i) for i in range(nbins)])

display(desi_binned_grads_df.head())
del nbins

In [ ]:
for i, (feature_mean, feature_std, label) in enumerate(zip(desi_gaston_grads, desi_gaston_stds, gaston_ion_labels_sorted)):
    if label in annot_labels:
        # Trajectory plot
        plot.disp_desi_gradient(feature_mean, feature_std, title=label)

        # DESI feature image
        plt.figure(figsize=(6, 6), dpi=100)
        plt.imshow(gaston_desi_img_sorted[i], cmap='magma')
        plt.title(label)
        plt.show()

        
del label, feature_mean, feature_std
del desi_gaston_grads, desi_gaston_stds, gaston_ion_labels_sorted

#### PCA

In [ ]:
adata_list = []
u_pc1s = []
u_pc2s = []
for desi_img in desi_imgs:
    desi_features = desi_img.transpose(2, 1, 0).reshape(-1, nchans)
    adata_x = sc.AnnData(desi_features)
    sc.pp.pca(adata_x, n_comps=5)

    u_pca = adata_x.obsm['X_pca']
    adata_list.append(adata_x)
    u_pc1s.append(u_pca[:, 0])
    u_pc2s.append(u_pca[:, 1])

u_pc1s_discrete = [utils.infer_zones(u.reshape(ndimy, ndimx).T, nbins=10) for u in u_pc1s]
u_pc2s_discrete = [utils.infer_zones(u.reshape(ndimy, ndimx).T, nbins=10) for u in u_pc2s]
del desi_img, desi_features, adata_x, u_pca

In [ ]:
plot.disp_chans([u.reshape(ndimy, ndimx).T for u in u_pc1s], cmap='turbo', title='PC1')
plot.disp_chans(u_pc1s_discrete, cmap='turbo', title='PC1 (discrete)')

In [ ]:
plot.disp_chans([u.reshape(ndimy, ndimx).T for u in u_pc2s], cmap='turbo', title='PC2')
plot.disp_chans(u_pc2s_discrete, cmap='turbo', title='PC2 (discrete)')

In [ ]:
# PCA w/ random shuffled channel ordering
u_pc1s_shuffled = []
for i, desi_img in enumerate(desi_imgs):
    desi_features = desi_img.transpose(2, 1, 0).reshape(-1, nchans)  # dim: [Y*X, C]
    desi_features = desi_features.T
    np.random.seed(i)
    np.random.shuffle(desi_features)
    adata_x = sc.AnnData(desi_features.T)  # dim: [Y*X, C]
    sc.pp.pca(adata_x, n_comps=5)

    u_pca = adata_x.obsm['X_pca']
    u_pc1s_shuffled.append(u_pca[:, 0])

u_pc1s_shuffled_discrete = [utils.infer_zones(u.reshape(ndimy, ndimx).T, nbins=10) for u in u_pc1s_shuffled]
del desi_img, desi_features, adata_x, u_pca 

In [ ]:
plot.disp_chans([u.reshape(ndimy, ndimx).T for u in u_pc1s_shuffled], cmap='turbo', title='PC1')
plot.disp_chans(u_pc1s_shuffled_discrete, cmap='turbo', title='PC1 (discrete)')

In [ ]:
desi_features_3d = np.array(desi_features_3d)
desi_features_unsorted = desi_features_3d.transpose(2, 0, 1).reshape(nchans, -1)  # Feature x Coord
desi_features_full = np.zeros_like(desi_features_unsorted)

indices = np.argsort(np.array(u_pcs).squeeze().flatten())
for i, chan in enumerate(desi_features_unsorted):
    desi_features_full[i] = chan[indices]

del desi_features_unsorted, indices, chan

In [ ]:
# Visualize DESI trajectory represented as PC0
plot.disp_desi_gradients(desi_features_full.T, 
                         ion_labels, 
                         title='PC1 isodepth', nbins=100)


In [ ]:
# Compute binned trajectory
nbins = 100
desi_pc_grads, desi_pc_stds = utils.get_binned_features(desi_features_full.T , nbins=nbins)  
desi_pc_grads, desi_pc_stds = desi_pc_grads.T, desi_pc_stds.T  # [C, #bins]

# Sort metabolites based on argmax values along the PV->CV binned trajectory
argmax_grad = desi_pc_grads.argmax(1)

ion_labels_sorted = np.array(ion_labels)[np.argsort(argmax_grad)]
desi_img_sorted = desi_imgs[idx][np.argsort(argmax_grad), ...]
desi_pc_grads = desi_pc_grads[np.argsort(argmax_grad), :]
desi_pc_stds = desi_pc_stds[np.argsort(argmax_grad), :]

desi_pc_grads_df = pd.DataFrame(desi_pc_grads,  
                                index=ion_labels_sorted,
                                columns=['bin' + str(i) for i in range(nbins)])

del nbins

In [ ]:
for i, (feature_mean, feature_std, label) in enumerate(zip(desi_pc_grads, desi_pc_stds, ion_labels_sorted)):
    if label in annot_labels:
        # Trajectory plot
        plot.disp_desi_gradient(feature_mean, feature_std, title=label + ' (PC1 3D)')
        
del label, feature_mean, feature_std
del desi_pc_grads, desi_pc_stds

In [ ]:
# Binned-expression correlations
g = sns.clustermap(desi_pc_grads_df.loc[annot_labels].T.corr(),
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n btw annotated DESI features', y=1.3, fontsize=20)
plt.show()

#### Clustering
e.g. K-means / Louvain / Leiden

In [ ]:
idx = 10
desi_features = desi_imgs[idx].transpose(2, 1, 0).reshape(-1, nchans)
adata_x = sc.AnnData(desi_features)

sc.pp.pca(adata_x)
sc.pp.neighbors(adata_x)
sc.tl.louvain(adata_x)
adata_x.obs.louvain = adata_x.obs.louvain.astype(np.uint8)

In [ ]:
# Use cluster assignment to represent spatial node modules
u_louvain = adata_x.obs.louvain.to_numpy().reshape(ndimy, ndimx).T
u_louvain_one_hot = [(u_louvain == i).astype(np.uint8)
                     for i in np.sort(adata_x.obs.louvain.unique())]  

plot.disp_chans(u_louvain_one_hot, cmap='magma')

#### LDA

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation as LDA

In [ ]:
idx = 10
desi_features = desi_imgs[idx].transpose(2, 1, 0).reshape(-1, nchans)
lda = LDA(n_components=5, random_state=42)
lda.fit(desi_features)

In [ ]:
theta = lda.transform(desi_features)
beta = lda.components_.T

In [ ]:
sns.clustermap(theta)

In [ ]:
sns.clustermap(beta)

In [ ]:
for i in range(lda.n_components):
    plt.figure()
    plt.imshow(theta[:, i].reshape(ndimy, ndimx).T, cmap='magma')
    plt.show()

In [ ]:
sns.clustermap(beta.T)

---